# Effect of pinholes on flux and apparent permeability

Temperature and pressure are held at `CONDITIONS['T_operating']` and `CONDITIONS['P_upstream']`, the oxide at `L_oxide` and the metal at `L_metal` (all from `study_config.py`). The pinhole area fraction $f$ is swept over `CONDITIONS['defect_fraction_range']`; cracks and oxide grain boundaries are set to zero.

**What a pinhole does.** A pinhole is a hole right through the oxide. Under it the gas meets bare metal: H₂ dissociates on the metal and the atoms enter it without ever crossing the oxide. The wall then has two paths side by side, intact oxide + metal and bare metal, which exchange no hydrogen, so it passes the area-weighted sum

$$J = (1-f)\,J_i + f\,J_p$$

$J_i$ and $J_p$ are the fluxes per unit area through intact oxide and through a pinhole, and neither depends on $f$. The pinhole path is the bare-metal level: L1 under L3, L4 under L5, L1+L6 under L3+L6. Section 2 checks both that and the no-pinhole limit.

**One number sets the curve: the intact oxide's permeation reduction factor**, $\mathrm{PRF} = J_p/J_i$, which is how much more one unit of bare metal passes than one unit of oxide-covered metal. The area fraction at which pinholes carry half the flux is

$$f_{1/2} = \frac{1}{1 + \mathrm{PRF}}$$

so the better the oxide, the fewer pinholes it takes to undo it: an oxide with PRF = 10⁶ is half bypassed at $f \approx 10^{-6}$.

**The slope.** For $J \propto f^{\,n}$ the local slope is $n(f) = f\,(J_p - J_i)/J$. It is 0 where the intact oxide controls and rises to 1 where the pinholes do. If bare metal passes less than oxide-covered metal (PRF < 1), $n$ is negative: pinholes lower the flux. $J$ is linear in $f$, so each regime's power-law fit returns the average of $n(f)$ over that regime's window, not a constant exponent.

**Apparent permeability.** $\Phi_{app} = J\,L_{total}/\Delta\sqrt{P}$, with $L_{total} = L_{ox} + L_m$, comes from the same flux. It is the permeability you would infer if you treated the pinholed wall as one uniform material. Thickness and pressure are fixed, so it moves exactly with $J$ and has the same $n$: pinholes raise the wall's apparent permeability by the factor $J/J_i$.

**Model 1 vs Model 2.** In Model 1 the oxide carries molecular H₂ ($J \propto P$) and the metal atomic H ($J \propto \sqrt{P}$), so the oxide holds hydrogen back only at low pressure. At high pressure PRF → 1 and pinholes change nothing. In Model 2 a pinhole also changes the surface: the bare metal dissociates H₂ with its own kinetics (`METALS[...]['surface_kinetics']`), the intact path with the oxide's. Where dissociation controls both paths, PRF is the ratio of the two dissociation rates, and pinholes let more hydrogen through only if the bare metal dissociates H₂ faster than the oxide does.

**Labels.** Each point names two things:
- the **dominant path**, whichever path carries more than half the flux: the intact oxide (filled markers) or the pinholes (hollow markers);
- the **limiting step** inside that path (colour): the oxide surface or oxide diffusion on the intact path, and the metal surface or metal diffusion on the pinhole path. A surface step is always the surface of its own path, so "oxide surface" means H₂ dissociation on the oxide and "metal surface" means H₂ dissociation on the bare metal under the pinholes.

The labels come from `calculations/wall_labels.py`, shared with the pressure, temperature and thickness notebooks. The limiting step is the solver's own classification of that path. In Model 2 it is the step holding more than half the path's resistance. In Model 1 it is oxide if $R_{ox}/R_m > 10$ and metal if $R_{ox}/R_m < 0.5$, and Model 1 has no surface step because its surface is at equilibrium. A path with no limiting step is *mixed*, and mixed points are drawn grey and not fitted. On each curve the label can change only at $f_{1/2}$, where the dominant path switches, because neither path depends on $f$.

Every point is also written to `*_paths.csv`: the flux through each path, its share, and which step (surface, oxide or metal) limits that path.

## 1. Setup

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import stats
from calculations.config.model_config import CONDITIONS, MICROSTRUCTURE, build_simulation_config
from calculations.oxide_permeation import get_oxide_properties_at_T, get_metal_properties_at_T
from calculations.permeation_calc import calculate_simple_metal_flux, calculate_defective_metal_flux
from calculations.interface_solver import calculate_oxide_metal_system, calculate_oxide_defective_metal_system
from calculations.parallel_oxide_defect_paths import (calculate_parallel_path_flux,
                                                      calculate_parallel_path_flux_defective_metal)
from calculations.permeability import UNITS_SIEVERTS
from calculations.wall_labels import model1_paths, model2_paths, label_from_paths, label_model2
import pandas as pd
from IPython.display import display
from calculations.figures import results_dir, save_figure
FIGS = results_dir("effect-of-pinholes")                              # figures, fit and path tables go in this folder

SIM          = build_simulation_config()
T_op, P_up   = CONDITIONS['T_operating'], CONDITIONS['P_upstream']
P_down       = SIM['P_downstream']
L_ox, L_m    = SIM['L_oxide'], SIM['L_metal']
F_GRID       = np.logspace(*np.log10(CONDITIONS['defect_fraction_range']), CONDITIONS['n_points'])   # pinhole area fraction
F_ZERO       = CONDITIONS['defect_limit_check']        # f -> 0: the no-pinhole limit of each curve
FRACTIONS    = ('fraction_surface', 'fraction_oxide', 'fraction_metal')

# Temperature is fixed, so the layer properties are computed once
OX = dict(get_oxide_properties_at_T(SIM['oxide_name'], T_op), thickness=L_ox)
ME = dict(get_metal_properties_at_T(SIM['metal_name'], T_op), thickness=L_m)
MICRO_M1 = dict(microstructure_params=MICROSTRUCTURE, lattice_density=MICROSTRUCTURE['N_L'], mode='both')

LEVELS = {   # level: (title, level the f -> 0 limit must reproduce, level the pinhole path must reproduce)
    'L3': ('Level 3: Defective oxide + perfect metal',   'L2b', 'L1'),
    'L5': ('Level 5: Defective oxide + defective metal', None,  'L4'),   # Model 1 has no perfect-oxide + defective-metal level
}

## 2. Physics (one solver call per point, then the sweep, its checks and the fit)

In [ ]:
def pinholes_only(f):
    '''Pinholes alone at area fraction f, as (Model 1 defect dict, Model 2 defect_config).

    Model 1 gets the single-type branch, never 'mixed': 'mixed' weights each component by
    component / sum and multiplies by a separate area_fraction, so it would re-weight the paths
    instead of adding area.
    '''
    return {'type': 'pinhole', 'area_fraction': f}, {'pinhole': {'area_fraction': f}}

def solve(level, f):
    '''One point: total flux, label, and each path's flux, contribution and limiting step (T and P fixed).'''
    if level.endswith('L6'):
        return solve_m2(level, f)                            # Model 2 levels, defined further down
    d = pinholes_only(f)[0]
    if level == 'L3':
        r = calculate_parallel_path_flux(P_up, P_down, OX, ME, d)
        paths = model1_paths(level, P_up, P_down, OX, ME, d)
    if level == 'L5':
        r = calculate_parallel_path_flux_defective_metal(P_up, P_down, OX, ME, d, T_op, **MICRO_M1)
        paths = model1_paths(level, P_up, P_down, OX, ME, d, T_op, MICRO_M1)
    return {'J': r['flux_total'], 'label': label_from_paths(paths, r['flux_total']), 'paths': paths}

def reference_flux(level):
    '''Flux of the level a limit is compared with (T and P fixed): the no-pinhole wall or the bare metal.'''
    if level == 'L1':
        return calculate_simple_metal_flux(ME['D_metal'], ME['K_s_metal'], L_m, P_up, P_down)['flux']
    if level == 'L2b':
        return calculate_oxide_metal_system(P_up, P_down, OX, ME)['flux']
    if level == 'L4':
        return calculate_defective_metal_flux(D_lattice=ME['D_metal'], K_s=ME['K_s_metal'], metal_thickness=L_m,
                                              P_up=P_up, P_down=P_down, temperature=T_op, **MICRO_M1)['flux']
    return solve_m2_reference(level)                         # Model 2, defined further down

NO_PATH = {'flux': np.nan, 'contribution': np.nan, 'step': 'undefined', 'detail': '',
           **{k: np.nan for k in FRACTIONS}}                # a path of a point whose solve failed

def sweep(level):
    '''One curve along F_GRID: J, Φ_app, label, the f -> 0 limit, the checks, and one paths-table row per point.'''
    pts = [solve(level, f) for f in F_GRID]
    J = np.array([p['J'] for p in pts], dtype=float)
    regime = np.array([p['label'] for p in pts])
    paths = [p['paths'] or {'intact': NO_PATH, 'pinhole': NO_PATH} for p in pts]
    per_path = lambda kind, key: np.array([q[kind][key] for q in paths], dtype=float)
    J_i, J_p = per_path('intact', 'flux'), per_path('pinhole', 'flux')
    ok = np.isfinite(J)
    assert ok.any(), f'{level}: every point failed to solve'
    assert np.all(ok | (regime == 'undefined')), f'{level}: a NaN flux without the undefined label'
    # the paths are uncoupled: each path flux is the same at every f, and the two contributions add up to J
    assert np.allclose(J_i[ok], J_i[ok][0], rtol=1e-9, atol=0) and np.allclose(J_p[ok], J_p[ok][0], rtol=1e-9, atol=0), \
        f'{level}: a path flux changes with f'
    assert np.allclose(per_path('intact', 'contribution')[ok] + per_path('pinhole', 'contribution')[ok], J[ok],
                       rtol=1e-9, atol=0), f'{level}: the path contributions do not add up to J'
    share_p = F_GRID * J_p / J
    assert np.all((share_p[ok] >= 0) & (share_p[ok] <= 1 + 1e-12)), f'{level}: pinhole share outside [0, 1]'

    # the two limits: f -> 0 is the no-pinhole wall, and the pinhole path is the bare-metal level
    J0 = solve(level, F_ZERO)['J']
    checks = {}
    for name, ours, ref in (('f→0', J0, LEVELS[level][1]), ('pinhole path', J_p[ok][0], LEVELS[level][2])):
        if ref:
            J_ref = reference_flux(ref)
            assert np.isclose(ours, J_ref, rtol=1e-6), f'{level}: {name} gives {ours:.6e}, {ref} gives {J_ref:.6e}'
            checks[f'{name} / {ref} − 1'] = ours / J_ref - 1

    to_phi = (L_ox + L_m) / (np.sqrt(P_up) - np.sqrt(P_down))   # every level here is Sieverts: Φ_app = J·L_total/Δ√P
    rows = []
    for f, Jt, ji, jp, sp, reg, q in zip(F_GRID, J, J_i, J_p, share_p, regime, paths):
        pi, pp = q['intact'], q['pinhole']
        rows.append({'level': level, 'T (K)': T_op, 'P_up (Pa)': P_up, 'f': f, 'label': reg,
                     'dominant_path': reg.split('/')[0], 'J_total': Jt, 'Phi_app': Jt * to_phi,
                     'J_intact': ji, 'contribution_intact': pi['contribution'], 'share_intact': 1 - sp,
                     'step_intact': pi['step'], 'metal_detail_intact': pi['detail'],
                     **{f'{k}_intact': pi[k] for k in FRACTIONS},
                     'J_pinhole': jp, 'contribution_pinhole': pp['contribution'], 'share_pinhole': sp,
                     'step_pinhole': pp['step'], 'metal_detail_pinhole': pp['detail'],
                     **{f'{k}_pinhole': pp[k] for k in FRACTIONS},
                     'PRF': jp / ji, 'f_half': ji / (ji + jp)})
    return {'J': J, 'Phi': J * to_phi, 'J0': J0, 'Phi0': J0 * to_phi, 'regime': regime,
            'rows': rows, 'checks': checks}

def power_law(f, y):
    '''y = A·f^n, fitted as ln y = ln A + n·ln f.'''
    fit = stats.linregress(np.log(f), np.log(y))
    return fit.slope, np.exp(fit.intercept), fit.rvalue**2   # n, A, R²

## 3. Style

In [ ]:
mpl.rcParams.update({
    'font.size': 14, 'font.weight': 'bold', 'mathtext.default': 'bf',
    'axes.labelweight': 'bold', 'axes.titleweight': 'bold', 'axes.linewidth': 1.8,
    'xtick.direction': 'in', 'ytick.direction': 'in', 'ytick.right': True,
    'xtick.major.width': 1.6, 'ytick.major.width': 1.6,
    'legend.frameon': False, 'legend.fontsize': 11, 'savefig.dpi': 600, 'savefig.bbox': 'tight',
})
INK, GREY, SHADE = '#0b0b0b', '#52514e', '#e1e0d9'

# Regime labels name the dominant path and the limiting step inside it (calculations/wall_labels.py):
# colour = limiting step, hollow = the pinholes carry most of the flux, diamond = trap-controlled.
from calculations.wall_labels import describe as look, fitted

UNITS_TEX = {UNITS_SIEVERTS: r'mol m$^{-1}$ s$^{-1}$ Pa$^{-1/2}$'}

## 4. Plot

In [ ]:
from matplotlib.lines import Line2D

LINES, ROLES = ['-', '--', ':'], ['low', 'mid', 'high']   # fit line style per condition, in order

def p_tag(Pc):
    '''Short pressure label: 1e-07 -> '1e-7 Pa', 1.0 -> '1e0 Pa', 1e+07 -> '1e7 Pa'.'''
    return f'{Pc:.0e}'.replace('e+0', 'e').replace('e-0', 'e-') + ' Pa'

def framed(nrows, figsize):
    '''Figure with two legend strips under the title and an nrows x 2 grid of panels below them.'''
    fig  = plt.figure(figsize=figsize, layout='constrained')
    gs   = fig.add_gridspec(nrows + 2, 2, height_ratios=[0.001, 0.001] + [1] * nrows)
    keys = [fig.add_subplot(gs[k, :]) for k in (0, 1)]
    for k in keys:
        k.axis('off')
    ax = np.empty((nrows, 2), dtype=object)
    for r in range(nrows):
        for c in range(2):
            ax[r, c] = fig.add_subplot(gs[r + 2, c], sharex=ax[0, 0] if (r or c) else None)
    return fig, keys, ax

def shared_legend(keys, axes, lines):
    '''One legend per figure: the markers (by regime) on the first strip, the lines on the second.'''
    marks = {}
    for a in axes:
        for h, label in zip(*a.get_legend_handles_labels()):
            marks.setdefault(label, h)
    keys[0].legend(list(marks.values()), list(marks), title='Simulation', loc='center', ncol=min(len(marks), 2))
    keys[1].legend(handles=lines, loc='center', ncol=len(lines))

def end_label(ax, xv, y, tag, ci):
    '''Name a curve at its right-hand end, staggered so coinciding curves stay readable.'''
    ok = np.flatnonzero(np.isfinite(y))
    i = ok[np.argmax(xv[ok])]
    ax.annotate(tag, (xv[i], y[i]), xytext=(12, 14 * (ci - 1)), textcoords='offset points',
                va='center', fontsize=11, color=INK)

def show(table):
    '''Display a table with compact numbers; returned unchanged for the CSV.'''
    table = table.replace({r'\$_2\$': '₂'}, regex=True)
    with pd.option_context('display.float_format', '{:.4g}'.format, 'display.width', 240,
                           'display.max_columns', 30):
        display(table)
    return table

EXPONENT = 'n (J, Φ_app ∝ f^n)'                         # L and P are fixed, so Φ_app and J share one exponent

def panel(ax, y, regime, rows, quantity, tag, ls):
    '''Regime-coloured simulation points with one power-law fit per regime; each fit becomes a table row.'''
    for reg in dict.fromkeys(regime):
        m = regime == reg
        name, c, mk, *fill = look(reg)
        hollow = fill == ['hollow']                      # Model 2: flux mainly through the pinholes
        ax.loglog(F_GRID[m], y[m], mk, ms=9, color=c, mfc='white' if hollow else c,
                  mec=c if hollow else 'white', mew=2 if hollow else 1, label=name)
        m &= np.isfinite(y) & (y > 0)
        if not fitted(reg) or m.sum() < 3:
            continue                                     # drawn, not fitted
        n, A, r2 = power_law(F_GRID[m], y[m])
        ax.loglog(F_GRID[m], A * F_GRID[m]**n, ls, lw=2.8, color=c)
        rows.append({'quantity': quantity, 'condition': tag, 'fit': name,
                     'f window': f'{F_GRID[m].min():.1e}–{F_GRID[m].max():.1e}', 'points': int(m.sum()),
                     EXPONENT: n, 'A (SI)': A, 'R²': r2})
    ax.grid(color=SHADE)

def pinhole_figure(level, curves=None, note=None):
    '''1x2 figure (J, Φ_app against pinhole area fraction), its fit table and its paths table.

    curves = [(tag, sweep output), ...], one per condition; by default the study's own (T_op, P_up).
    '''
    title = LEVELS[level][0]
    curves = curves or [(f'{T_op:.0f} K, {p_tag(P_up)}', sweep(level))]
    multi = len(curves) > 1
    fig, keys, ax = framed(1, (14, 8.5))
    fit_rows, path_rows, summary = [], [], []
    for ci, (tag, out) in enumerate(curves):
        ls = LINES[ci] if multi else '-'
        for j, (qty, y, y0) in enumerate((('J', out['J'], out['J0']), ('Phi_app', out['Phi'], out['Phi0']))):
            p = ax[0, j]
            panel(p, y, out['regime'], fit_rows, qty, tag, ls)
            p.axhline(y0, color=GREY, lw=1.2, ls=ls, alpha=0.6, zorder=0)        # f -> 0: no pinholes
            if multi:
                end_label(p, F_GRID, y, tag, ci)
        path_rows += out['rows']
        first = next(r for r in out['rows'] if np.isfinite(r['J_total']))
        summary.append({'condition': tag, 'J_intact': first['J_intact'], 'J_pinhole': first['J_pinhole'],
                        'PRF = J_p/J_i': first['PRF'], 'f½': first['f_half'],
                        'intact path limited by': ' '.join(filter(None, (first['step_intact'], first['metal_detail_intact']))),
                        'pinhole path limited by': ' '.join(filter(None, (first['step_pinhole'], first['metal_detail_pinhole']))),
                        **out['checks']})
    for j, ylab in enumerate((r'Flux $J$ (mol m$^{-2}$ s$^{-1}$)', rf'$\Phi_{{app}}$ ({UNITS_TEX[UNITS_SIEVERTS]})')):
        ax[0, j].set_xlabel('Pinhole area fraction $f$')
        ax[0, j].set_ylabel(ylab)
    ax[0, 0].set_title('Flux')
    ax[0, 1].set_title('Apparent permeability')
    ax[0, 0].set_xlim(F_GRID.min() / 1.5, F_GRID.max() * (5 if multi else 1.5))   # room for the end labels
    if multi:
        lines = [Line2D([], [], color=INK, lw=2.4, ls=LINES[ci], label=f'{ROLES[ci]}: {tag}')
                 for ci, (tag, _) in enumerate(curves)]
    else:
        lines = [Line2D([], [], color=INK, lw=2.4, label='Power-law fit (colour: per regime)')]
    lines.append(Line2D([], [], color=GREY, lw=1.2, alpha=0.6, label='No pinholes (f → 0)'))
    shared_legend(keys, ax.flat, lines)
    where = note or rf'$T$ = {T_op:.0f} K,   $P_{{up}}$ = {P_up:.0e} Pa'
    fig.suptitle(f'{title}, pinholes only\n{where}\n'
                 + rf'$J = (1-f)\,J_i + f\,J_p$,   $\Phi_{{app}} = J\,L_{{total}}/\Delta\sqrt{{P}}$,   $L_{{total}}$ = {L_ox + L_m:.2e} m',
                 fontsize=16, fontweight='bold')
    plt.show()
    show(pd.DataFrame(summary))
    return fig, show(pd.DataFrame(fit_rows)), pd.DataFrame(path_rows)

## 5. Model 1 (L3, L5)

In [ ]:
for level in LEVELS:
    fig, fits, paths = pinhole_figure(level)
    save_figure(fig, f"{level}_pinholes", FIGS)
    fits.to_csv(FIGS / f"{level}_pinholes_fits.csv", index=False)
    paths.to_csv(FIGS / f"{level}_pinholes_paths.csv", index=False)

## 6. Model 2 (L3+L6, L5+L6)

In [ ]:
# =============================================================================
# MODEL 2 (L6 family): dissociation on the gas-facing surface, atomic H in oxide and metal
# Reuses solve() -> sweep() -> pinhole_figure() above; only the solver is new.
# The pinhole exposes the bare metal, so its path dissociates H2 with the metal's own surface
# kinetics (SM); the intact path uses the oxide surface (PROPS).
# =============================================================================
from calculations.surface_kinetics import (get_all_properties, solve_steady_state_flux_L1L6,
                                           solve_steady_state_flux_direct, calculate_mixed_defect_flux_L6,
                                           calculate_defective_metal_flux_L6, calculate_full_model_flux_L346_v2)
from calculations.config.model_config import METALS, get_surface_kinetics_from_config

MICRO = SIM['microstructure']
SIEVERTS_PINHOLE = SIM['oxide_defects'].get('use_sieverts_pinhole', False)

# Temperature is fixed, so the Model 2 properties are computed once
PROPS = get_all_properties(SIM['oxide_name'], SIM['metal_name'], T_op)       # oxide surface + transport
SM    = get_surface_kinetics_from_config(SIM['metal_name'], T_op, METALS)    # bare-metal surface

LEVELS_M2 = {   # level: (title, level the f -> 0 limit must reproduce, level the pinhole path must reproduce)
    'L3L6': ('Level 3+6: Defective oxide + perfect metal + surface kinetics',   'L2bL6', 'L1' if SIEVERTS_PINHOLE else 'L1L6'),
    'L5L6': ('Level 5+6: Defective oxide + defective metal + surface kinetics', 'L4L6',  None),   # no bare defective-metal + surface level
}
LEVELS.update(LEVELS_M2)

def solve_m2(level, f):
    '''One Model 2 point: total flux, label, and each path's flux, contribution and limiting step.'''
    k, Keq, Dox, Kox, Dm, Ksm = (PROPS[key] for key in ('k_diss', 'K_eq', 'D_ox', 'K_ox', 'D_m', 'K_s_m'))
    pin  = {} if SIEVERTS_PINHOLE else dict(k_diss_metal=SM['k_diss'], K_eq_metal=SM['K_eq'])
    defects = pinholes_only(f)[1]
    if level == 'L3L6':  r = calculate_mixed_defect_flux_L6(P_up, P_down, L_m, k, Keq, Dox, Kox, L_ox,
                                                            Dm, Ksm, defects, **pin)
    if level == 'L5L6':  r = calculate_full_model_flux_L346_v2(P_up, P_down, L_m, T_op, k, Keq, Dox, Kox, L_ox,
                                                               Dm, Ksm, MICRO, defects, mode=MICRO.get('mode', 'both'), **pin)
    label = label_model2(level, r, defects)                 # 'undefined' if the pinhole path was dropped
    if label == 'undefined':
        return {'J': np.nan, 'label': label, 'paths': None}
    return {'J': r['J_total'], 'label': label, 'paths': model2_paths(level, r)}

def solve_m2_reference(level):
    '''Flux of the Model 2 level a limit is compared with.'''
    k, Keq, Dox, Kox, Dm, Ksm = (PROPS[key] for key in ('k_diss', 'K_eq', 'D_ox', 'K_ox', 'D_m', 'K_s_m'))
    if level == 'L1L6':  r = solve_steady_state_flux_L1L6(P_up, P_down, L_m, SM['k_diss'], SM['K_eq'], Dm, Ksm)
    if level == 'L2bL6': r = solve_steady_state_flux_direct(P_up, P_down, L_m, k, Keq, Dox, Kox, L_ox, Dm, Ksm)
    if level == 'L4L6':  r = calculate_defective_metal_flux_L6(P_up, P_down, L_m, T_op, k, Keq, Dox, Kox, L_ox,
                                                               Dm, Ksm, microstructure_params=MICRO,
                                                               mode=MICRO.get('mode', 'both'))
    return r.get('J_total', r.get('J_ss', r.get('flux')))

for level in LEVELS_M2:
    fig, fits, paths = pinhole_figure(level)
    save_figure(fig, f"{level}_pinholes", FIGS)
    fits.to_csv(FIGS / f"{level}_pinholes_fits.csv", index=False)
    paths.to_csv(FIGS / f"{level}_pinholes_paths.csv", index=False)

## 7. Three temperatures and three pressures: the pinhole sweep at the low / mid / high of each range

Same figure as above, with one curve per condition in each panel. Colour and marker still mean the regime; line style and the label at the end of each curve mean the condition.

In [ ]:
# =============================================================================
# THREE TEMPERATURES (at P_up) and THREE PRESSURES (at T_op): low / mid / high of each range
# =============================================================================
from contextlib import contextmanager

T_lo, T_hi = CONDITIONS['T_range']
P_lo, P_hi = CONDITIONS['P_range']
SETS = {   # set: (title note, [(curve label, at_conditions arguments), ...])
    'T': ('three temperatures (low / mid / high of T_range),   ' + rf'$P_{{up}}$ = {P_up:.0e} Pa',
          [(f'{Tc:.0f} K', dict(Tc=Tc)) for Tc in (T_lo, (T_lo + T_hi) / 2, T_hi)]),
    'P': ('three pressures (low / mid / high of P_range),   ' + rf'$T$ = {T_op:.0f} K',
          [(p_tag(Pc), dict(Pc=Pc)) for Pc in (P_lo, np.sqrt(P_lo * P_hi), P_hi)]),
}

@contextmanager
def at_conditions(Tc=None, Pc=None):
    '''Run solve() and sweep() above at another (T, P_up); the study values are restored afterwards.'''
    global T_op, P_up, OX, ME, PROPS, SM
    saved = T_op, P_up, OX, ME, PROPS, SM
    T_op  = T_op if Tc is None else Tc
    P_up  = P_up if Pc is None else Pc
    OX    = dict(get_oxide_properties_at_T(SIM['oxide_name'], T_op), thickness=L_ox)
    ME    = dict(get_metal_properties_at_T(SIM['metal_name'], T_op), thickness=L_m)
    PROPS = get_all_properties(SIM['oxide_name'], SIM['metal_name'], T_op)
    SM    = get_surface_kinetics_from_config(SIM['metal_name'], T_op, METALS)
    try:
        yield
    finally:
        T_op, P_up, OX, ME, PROPS, SM = saved

for level in LEVELS:
    for cond in SETS:
        note, conds = SETS[cond]
        curves = []
        for tag, kw in conds:
            with at_conditions(**kw):
                curves.append((tag, sweep(level)))
        fig, fits, paths = pinhole_figure(level, curves, note)
        save_figure(fig, f"{level}_pinholes_3{cond}", FIGS)
        fits.to_csv(FIGS / f"{level}_pinholes_3{cond}_fits.csv", index=False)
        paths.to_csv(FIGS / f"{level}_pinholes_3{cond}_paths.csv", index=False)